# Main 04 — longitudinal paired and clinical analyses

Run iTBS, rTMS, and sham paired analyses plus prespecified clinical correlations.

- Formal analysis universe: 12 cortical + 14 limbic ROIs (26 nodes)
- Cross-network connectivity: 12 × 14 = 168 edges
- Pipeline: `main`; step 4 of 10
- Run mode: standalone; all prerequisite steps load automatically in the same kernel.


In [1]:
from pathlib import Path
import os

_repo = Path(os.environ.get("METAFL_ROOT", Path.cwd())).resolve()
if not (_repo / "pipeline_steps").is_dir() and (_repo.parent / "pipeline_steps").is_dir():
    _repo = _repo.parent
if not (_repo / "pipeline_steps").is_dir():
    raise FileNotFoundError("Set METAFL_ROOT to the repository root")
os.environ["METAFL_ROOT"] = str(_repo)
os.chdir(_repo)

def _run_step(relative_path):
    _path = _repo / "pipeline_steps" / relative_path
    _code = compile(_path.read_text(encoding="utf-8"), str(_path), "exec")
    exec(_code, get_ipython().user_ns)
_run_step("main_01_setup_load.py")
_run_step("main_02_demographics_hc_reference.py")
_run_step("main_03_baseline_discovery_sensitivity.py")


MetaFL package root: /Users/freddylee/Desktop/Coding_repository/Python_analysis/metabolism_TRD_release_refresh_20261007
Main-analysis output root: /Users/freddylee/Desktop/Coding_repository/Python_analysis/metabolism_TRD_release_refresh_20261007/outputs/main_analyses
Cortical IDs: [3, 4, 5, 6, 19, 20, 21, 22, 25, 26, 31, 32]
Limbic IDs: [41, 42, 43, 44, 45, 46, 75, 76, 77, 78, 79, 80, 151, 152]
Cross-edge family: 168


,cohort,clinical_n,ses01_complete_n
0,HC,23,23
1,nTRD,25,25
2,TRD1,35,35
3,TRD2,85,85


,cohort,baseline_n,metric
0,nTRD,25,age/sex-adjusted normative-SC MwC deviation z
1,TRD1,35,age/sex-adjusted normative-SC MwC deviation z
2,TRD2,85,age/sex-adjusted normative-SC MwC deviation z


Written: /Users/freddylee/Desktop/Coding_repository/Python_analysis/metabolism_TRD_release_refresh_20261007/outputs/main_analyses/demographic_and_clinical_summary.xlsx


,variable,HC,nTRD,TRD1,TRD2,primary_test,primary_p_value,TRD1_vs_TRD2_test,TRD1_vs_TRD2_p_value
0,Age,50.91 (17.04); n=23,52.32 (12.22); n=25,48.20 (15.52); n=35,38.40 (14.43); n=85,"one-way ANOVA: HC, nTRD, TRD1",0.561396,Welch t-test,0.002155
1,"Female, n (%)",14 (60.87%); n=23,14 (56.00%); n=25,25 (71.43%); n=35,54 (63.53%); n=85,"Pearson chi-square: HC, nTRD, TRD1",0.443311,Fisher exact test,0.525990
2,Education,14.17 (3.46); n=23,13.44 (3.86); n=25,13.14 (3.83); n=35,14.49 (2.97); n=85,"one-way ANOVA: HC, nTRD, TRD1",0.587829,Welch t-test,0.067189


...

,variable,note
0,MSM scores,Not required for the released analyses


,variable,iTBS,rTMS,sham,test,statistic,df_1,df_2,p_value
0,HDRS reduction (baseline - post),7.10 (8.18); n=31,8.86 (6.67); n=28,6.23 (5.17); n=26,ordinary one-way ANOVA,1.035219,2,82,0.359743


,variable,rTMS,iTBS,sham,test,p_value
0,"Age, years",38.29 (11.61); n=28,37.35 (15.40); n=31,39.77 (16.32); n=26,one-way ANOVA,0.822937
1,"Female, n (%)",17 (60.71%); n=28,20 (64.52%); n=31,17 (65.38%); n=26,Pearson chi-square,0.928941
2,"Education, years",14.64 (2.64); n=28,14.06 (2.68); n=31,14.85 (3.61); n=26,one-way ANOVA,0.586351
3,Baseline HDRS-17,22.46 (4.88); n=28,23.19 (5.35); n=31,22.81 (5.48); n=26,one-way ANOVA,0.866991
4,HDRS-17 reduction,8.86 (6.67); n=28,7.10 (8.18); n=31,6.23 (5.17); n=26,one-way ANOVA,0.359743


,modality,tested_n,primary_p_lt_0p01,HDRS_adjusted_p_lt_0p01,retained,lost,new
0,edge,168,19,2,1,18,1
1,regional,26,8,3,3,5,0
2,MwC,26,3,0,0,3,0


TRD2 subject-level deviations: (85, 168) (85, 26)


## Preamble


In [2]:
"""Auto-extracted pipeline step: Main 04 — longitudinal paired tests and clinical associations.

Do not edit generated copies independently of the source notebooks.
"""


'Auto-extracted pipeline step: Main 04 — longitudinal paired tests and clinical associations.\n\nDo not edit generated copies independently of the source notebooks.\n'

## Source 21


In [3]:
def add_hdrs_correlations(selected, value_matrix, feature_lookup, hdrs, age, sex, q_column):
    rows = []
    for _, selected_row in selected.iterrows():
        index = feature_lookup(selected_row)
        r_value, p_value, n_used = partial_spearman_age_sex(
            value_matrix[:, index], hdrs, age, sex
        )
        row = selected_row.to_dict()
        row.update({
            "HDRS_partial_Spearman_r": r_value,
            "HDRS_p_uncorrected": p_value,
            "n_used": n_used,
        })
        rows.append(row)

    columns = list(selected.columns) + [
        "HDRS_partial_Spearman_r", "HDRS_p_uncorrected", "n_used", q_column,
        "HDRS_primary_FDR_q_lt_0p05", "HDRS_exploratory_uncorrected_p_lt_0p05",
    ]
    if not rows:
        return pd.DataFrame(columns=columns)

    result = pd.DataFrame(rows)
    result[q_column] = bh_fdr(result["HDRS_p_uncorrected"])
    result["HDRS_primary_FDR_q_lt_0p05"] = result[q_column] < FDR_THRESHOLD
    result["HDRS_exploratory_uncorrected_p_lt_0p05"] = (
        result["HDRS_p_uncorrected"] < CLINICAL_P_THRESHOLD
    )
    return result.sort_values([q_column, "HDRS_p_uncorrected"]).reset_index(drop=True)


def save_prepost_and_hdrs(all_tests, selected, hdrs_results, output_dir, stem, q_column, include_hdrs=True):
    all_tests.to_excel(output_dir / f"{stem}_pairttest_all.xlsx", index=False)
    selected.to_excel(
        output_dir / f"{stem}_pairttest_uncorrected_p_lt_0p05.xlsx", index=False
    )
    hdrs_fdr = hdrs_results[hdrs_results["HDRS_primary_FDR_q_lt_0p05"]].copy()
    hdrs_exploratory = hdrs_results[
        hdrs_results["HDRS_exploratory_uncorrected_p_lt_0p05"]
    ].copy()
    if include_hdrs:
        hdrs_results.to_excel(
            output_dir / f"{stem}_HDRS_partial_spearman_all_prepost_p_lt_0p05.xlsx",
            index=False,
        )
        hdrs_fdr.to_excel(
            output_dir / f"{stem}_HDRS_partial_spearman_FDR_q_lt_0p05_primary.xlsx",
            index=False,
        )
        hdrs_exploratory.to_excel(
            output_dir / f"{stem}_HDRS_partial_spearman_uncorrected_p_lt_0p05_exploratory.xlsx",
            index=False,
        )
    return hdrs_fdr, hdrs_exploratory


def paired_mean_ci95(delta):
    delta = np.asarray(delta, dtype=float)
    n = np.sum(np.isfinite(delta), axis=0)
    mean = np.nanmean(delta, axis=0)
    sem = stats.sem(delta, axis=0, nan_policy="omit")
    critical = stats.t.ppf(0.975, np.maximum(n - 1, 1))
    return mean - critical * sem, mean + critical * sem


def run_arm_prepost(arm, context, output_dir, include_hdrs=True):
    output_dir.mkdir(parents=True, exist_ok=True)
    arm_clinical = clinical[
        (clinical["analysis_cohort"] == "TRD2")
        & (clinical["analysis_arm"] == arm)
    ].copy()
    required = ROI_COLUMNS + ["age", "sex", "education", "hdrs_improvement"]
    pre = arm_clinical.merge(pet_pre, on="subject_key", how="inner").dropna(subset=required)
    post = arm_clinical.merge(pet_post, on="subject_key", how="inner").dropna(subset=required)
    paired = sorted(set(pre["subject_key"]).intersection(post["subject_key"]))
    pre = pre[pre["subject_key"].isin(paired)].sort_values("subject_key").reset_index(drop=True)
    post = post[post["subject_key"].isin(paired)].sort_values("subject_key").reset_index(drop=True)
    assert len(pre) == EXPECTED_PAIRED_ARMS[arm]
    assert pre["subject_key"].tolist() == post["subject_key"].tolist()

    edge_pre = individual_edge_deviation(
        pre,
        context["reference_values"],
        context["reference_covariates"],
        context["pcc_reference"],
        context["denominator"],
        context["reference_covariate_names"],
    )
    edge_post = individual_edge_deviation(
        post,
        context["reference_values"],
        context["reference_covariates"],
        context["pcc_reference"],
        context["denominator"],
        context["reference_covariate_names"],
    )
    edge_delta = edge_post - edge_pre
    t_edge, p_edge = stats.ttest_rel(edge_post, edge_pre, axis=0, nan_policy="omit")
    edge_ci_low, edge_ci_high = paired_mean_ci95(edge_delta)

    edge_all = pd.DataFrame({
        "ROI_1": [pair[0] for pair in CROSS_EDGE_NAMES],
        "ROI_2": [pair[1] for pair in CROSS_EDGE_NAMES],
        "pre_deviation": np.nanmean(edge_pre, axis=0),
        "pre_SD_deviation": np.nanstd(edge_pre, axis=0, ddof=1),
        "post_deviation": np.nanmean(edge_post, axis=0),
        "post_SD_deviation": np.nanstd(edge_post, axis=0, ddof=1),
        "post_minus_pre": np.nanmean(edge_delta, axis=0),
        "post_minus_pre_CI95_low": edge_ci_low,
        "post_minus_pre_CI95_high": edge_ci_high,
        "paired_t": t_edge,
        "effect_size_Cohen_dz": t_edge / np.sqrt(len(pre)),
        "paired_p_uncorrected": p_edge,
    }).sort_values("paired_p_uncorrected").reset_index(drop=True)
    edge_selected = edge_all[
        edge_all["paired_p_uncorrected"] < PREPOST_P_THRESHOLD
    ].copy()

    hdrs = pre["hdrs_improvement"].to_numpy(float)
    age = pre["age"].to_numpy(float)
    sex = pre["sex"].to_numpy(float)
    edge_lookup = {pair: index for index, pair in enumerate(CROSS_EDGE_NAMES)}
    edge_hdrs_all = add_hdrs_correlations(
        edge_selected if include_hdrs else edge_selected.iloc[0:0],
        edge_delta,
        lambda row: edge_lookup[(row["ROI_1"], row["ROI_2"])],
        hdrs,
        age,
        sex,
        "HDRS_q_fdr_BH_within_prepost_significant_edges",
    )
    edge_hdrs_fdr, edge_hdrs_exploratory = save_prepost_and_hdrs(
        edge_all,
        edge_selected,
        edge_hdrs_all,
        output_dir,
        "edge",
        "HDRS_q_fdr_BH_within_prepost_significant_edges",
        include_hdrs=include_hdrs,
    )

    pre_design = sm.add_constant(pre[["age", "sex"]], has_constant="add").reindex(
        columns=context["regional_hc_design_columns"]
    )
    post_design = sm.add_constant(post[["age", "sex"]], has_constant="add").reindex(
        columns=context["regional_hc_design_columns"]
    )
    regional_pre = (
        pre[ROI_COLUMNS].to_numpy(float) - pre_design.to_numpy(float) @ context["regional_beta"]
    ) / context["reference_sd"].to_numpy(float)
    regional_post = (
        post[ROI_COLUMNS].to_numpy(float) - post_design.to_numpy(float) @ context["regional_beta"]
    ) / context["reference_sd"].to_numpy(float)
    regional_delta = regional_post - regional_pre
    t_regional, p_regional = stats.ttest_rel(
        regional_post, regional_pre, axis=0, nan_policy="omit"
    )
    regional_ci_low, regional_ci_high = paired_mean_ci95(regional_delta)
    regional_all = pd.DataFrame({
        "ROI": ROI_COLUMNS,
        "pre_deviation": np.nanmean(regional_pre, axis=0),
        "pre_SD_deviation": np.nanstd(regional_pre, axis=0, ddof=1),
        "post_deviation": np.nanmean(regional_post, axis=0),
        "post_SD_deviation": np.nanstd(regional_post, axis=0, ddof=1),
        "post_minus_pre": np.nanmean(regional_delta, axis=0),
        "post_minus_pre_CI95_low": regional_ci_low,
        "post_minus_pre_CI95_high": regional_ci_high,
        "paired_t": t_regional,
        "effect_size_Cohen_dz": t_regional / np.sqrt(len(pre)),
        "paired_p_uncorrected": p_regional,
    }).sort_values("paired_p_uncorrected").reset_index(drop=True)
    regional_selected = regional_all[
        regional_all["paired_p_uncorrected"] < PREPOST_P_THRESHOLD
    ].copy()
    regional_lookup = {roi: index for index, roi in enumerate(ROI_COLUMNS)}
    regional_hdrs_all = add_hdrs_correlations(
        regional_selected if include_hdrs else regional_selected.iloc[0:0],
        regional_delta,
        lambda row: regional_lookup[row["ROI"]],
        hdrs,
        age,
        sex,
        "HDRS_q_fdr_BH_within_prepost_significant_regions",
    )
    regional_hdrs_fdr, regional_hdrs_exploratory = save_prepost_and_hdrs(
        regional_all,
        regional_selected,
        regional_hdrs_all,
        output_dir,
        "regional",
        "HDRS_q_fdr_BH_within_prepost_significant_regions",
        include_hdrs=include_hdrs,
    )

    return {
        "arm": arm,
        "n": len(pre),
        "edge_all": edge_all,
        "edge_selected": edge_selected,
        "edge_hdrs_all": edge_hdrs_all,
        "edge_hdrs_fdr": edge_hdrs_fdr,
        "edge_hdrs_exploratory": edge_hdrs_exploratory,
        "regional_all": regional_all,
        "regional_selected": regional_selected,
        "regional_hdrs_all": regional_hdrs_all,
        "regional_hdrs_fdr": regional_hdrs_fdr,
        "regional_hdrs_exploratory": regional_hdrs_exploratory,
        "subject_keys": pre["subject_key"].to_numpy(),
        "hdrs_improvement": hdrs,
        "age": age,
        "sex": sex,
        "education": pre["education"].to_numpy(float),
        "edge_pre": edge_pre,
        "edge_post": edge_post,
        "edge_delta": edge_delta,
        "regional_pre": regional_pre,
        "regional_post": regional_post,
        "regional_delta": regional_delta,
    }


itbs_result = run_arm_prepost("iTBS", trd2_context, TRD2_OUT / "03_iTBS_prepost")
rtms_result = run_arm_prepost("rTMS", trd2_context, TRD2_OUT / "04_rTMS_prepost")
sham_result = run_arm_prepost("sham", trd2_context, TRD2_OUT / "05_sham_prepost", include_hdrs=False)

LONGITUDINAL_EDGE_FILE = TRD2_OUT / "longitudinal_high_quality_PET_subject_matrices_cross168.npz"
_paired_results = [itbs_result, rtms_result, sham_result]
np.savez_compressed(
    LONGITUDINAL_EDGE_FILE,
    subject_keys=np.concatenate([result["subject_keys"] for result in _paired_results]),
    arms=np.concatenate([np.repeat(result["arm"], result["n"]) for result in _paired_results]),
    edge_pre=np.vstack([result["edge_pre"] for result in _paired_results]),
    edge_post=np.vstack([result["edge_post"] for result in _paired_results]),
    edge_roi1=np.asarray([edge[0] for edge in CROSS_EDGE_NAMES], dtype=object),
    edge_roi2=np.asarray([edge[1] for edge in CROSS_EDGE_NAMES], dtype=object),
)
assert sum(EXPECTED_PAIRED_ARMS.values()) == sum(result["n"] for result in _paired_results)
print("Longitudinal edge archive:", LONGITUDINAL_EDGE_FILE)


prepost_counts = pd.DataFrame([
    {
        "arm": result["arm"],
        "n": result["n"],
        "edge_prepost_uncorrected_p_lt_0p05": len(result["edge_selected"]),
        "edge_HDRS_FDR_q_lt_0p05_primary": len(result["edge_hdrs_fdr"]),
        "edge_HDRS_uncorrected_p_lt_0p05_exploratory": len(result["edge_hdrs_exploratory"]),
        "regional_prepost_uncorrected_p_lt_0p05": len(result["regional_selected"]),
        "regional_HDRS_FDR_q_lt_0p05_primary": len(result["regional_hdrs_fdr"]),
        "regional_HDRS_uncorrected_p_lt_0p05_exploratory": len(result["regional_hdrs_exploratory"]),
    }
    for result in [itbs_result, rtms_result, sham_result]
])
prepost_counts.to_excel(TRD2_OUT / "prepost_and_HDRS_summary_counts.xlsx", index=False)
display(prepost_counts)


Longitudinal edge archive: /Users/freddylee/Desktop/Coding_repository/Python_analysis/metabolism_TRD_release_refresh_20261007/outputs/main_analyses/02_TRD2_deviation_prepost_agesex/longitudinal_high_quality_PET_subject_matrices_cross168.npz


,arm,n,edge_prepost_uncorrected_p_lt_0p05,edge_HDRS_FDR_q_lt_0p05_primary,edge_HDRS_uncorrected_p_lt_0p05_exploratory,regional_prepost_uncorrected_p_lt_0p05,regional_HDRS_FDR_q_lt_0p05_primary,regional_HDRS_uncorrected_p_lt_0p05_exploratory
0,iTBS,31,15,0,0,0,0,0
1,rTMS,28,13,0,2,0,0,0
2,sham,26,1,0,0,3,0,0


## Results preview

Key numerical tables and figures generated by this step are displayed below. The full-resolution files remain under `outputs/`.


In [4]:
from io import BytesIO
import json as _preview_json
import pandas as _preview_pd
from PIL import Image as PILImage
from IPython.display import Image as NotebookImage, Markdown, display

_inline_table_specs = [{'title': 'Longitudinal sample and result counts', 'path': 'outputs/main_analyses/02_TRD2_deviation_prepost_agesex/prepost_and_HDRS_summary_counts.xlsx'}, {'title': 'iTBS connectivity changes used for figures (paired p < .01)', 'path': 'outputs/main_analyses/02_TRD2_deviation_prepost_agesex/03_iTBS_prepost/edge_pairttest_all.xlsx', 'query': 'paired_p_uncorrected < 0.01'}, {'title': 'rTMS connectivity changes used for figures (paired p < .01)', 'path': 'outputs/main_analyses/02_TRD2_deviation_prepost_agesex/04_rTMS_prepost/edge_pairttest_all.xlsx', 'query': 'paired_p_uncorrected < 0.01'}, {'title': 'iTBS HDRS-17 associations', 'path': 'outputs/main_analyses/02_TRD2_deviation_prepost_agesex/03_iTBS_prepost/edge_HDRS_partial_spearman_all_prepost_p_lt_0p05.xlsx', 'max_rows': 20}, {'title': 'rTMS HDRS-17 associations', 'path': 'outputs/main_analyses/02_TRD2_deviation_prepost_agesex/04_rTMS_prepost/edge_HDRS_partial_spearman_all_prepost_p_lt_0p05.xlsx', 'max_rows': 20}]
_inline_figure_paths = []

display(Markdown("### Key tables"))
for _table_spec in _inline_table_specs:
    _relative_table = _table_spec["path"]
    _table_path = _repo / _relative_table
    if not _table_path.is_file():
        raise FileNotFoundError(_table_path)
    if _table_path.suffix.lower() == ".csv":
        _preview_table = _preview_pd.read_csv(_table_path)
    elif _table_path.suffix.lower() in {".xlsx", ".xls"}:
        _preview_table = _preview_pd.read_excel(
            _table_path, sheet_name=_table_spec.get("sheet", 0)
        )
    elif _table_path.suffix.lower() == ".json":
        _payload = _preview_json.loads(_table_path.read_text(encoding="utf-8"))
        _checks = _payload.get("checks", _payload)
        _preview_table = _preview_pd.DataFrame(
            {"item": list(_checks), "value": list(_checks.values())}
        ) if isinstance(_checks, dict) else _preview_pd.json_normalize(_payload)
    else:
        raise ValueError(f"Unsupported table format: {_table_path.suffix}")
    if _table_spec.get("query"):
        _preview_table = _preview_table.query(_table_spec["query"]).reset_index(drop=True)
    _max_rows = int(_table_spec.get("max_rows", 40))
    display(Markdown(
        f"**{_table_spec['title']}**  \n"
        f"`{_relative_table}` — {len(_preview_table)} rows × {len(_preview_table.columns)} columns"
    ))
    with _preview_pd.option_context("display.max_columns", 40, "display.max_colwidth", 60):
        display(_preview_table.head(_max_rows))

if _inline_figure_paths:
    display(Markdown("### Figures"))
for _relative_figure in _inline_figure_paths:
    _figure_path = _repo / _relative_figure
    if not _figure_path.is_file():
        raise FileNotFoundError(_figure_path)
    with PILImage.open(_figure_path) as _source_image:
        _preview_image = _source_image.convert("RGB")
        _preview_image.thumbnail((1600, 1200), PILImage.Resampling.LANCZOS)
        _preview_buffer = BytesIO()
        _preview_image.save(_preview_buffer, format="PNG", optimize=True)
    display(Markdown(f"**`{_relative_figure}`**"))
    display(NotebookImage(data=_preview_buffer.getvalue(), format="png"))


### Key tables

**Longitudinal sample and result counts**  
`outputs/main_analyses/02_TRD2_deviation_prepost_agesex/prepost_and_HDRS_summary_counts.xlsx` — 3 rows × 8 columns

,arm,n,edge_prepost_uncorrected_p_lt_0p05,edge_HDRS_FDR_q_lt_0p05_primary,edge_HDRS_uncorrected_p_lt_0p05_exploratory,regional_prepost_uncorrected_p_lt_0p05,regional_HDRS_FDR_q_lt_0p05_primary,regional_HDRS_uncorrected_p_lt_0p05_exploratory
0,iTBS,31,15,0,0,0,0,0
1,rTMS,28,13,0,2,0,0,0
2,sham,26,1,0,0,3,0,0


**iTBS connectivity changes used for figures (paired p < .01)**  
`outputs/main_analyses/02_TRD2_deviation_prepost_agesex/03_iTBS_prepost/edge_pairttest_all.xlsx` — 2 rows × 12 columns

,ROI_1,ROI_2,pre_deviation,pre_SD_deviation,post_deviation,post_SD_deviation,post_minus_pre,post_minus_pre_CI95_low,post_minus_pre_CI95_high,paired_t,effect_size_Cohen_dz,paired_p_uncorrected
0,AAL3_031_OFClat_L,AAL3_041_Hippocampus_L,-0.194847,0.644053,-0.482488,0.819397,-0.287641,-0.450633,-0.124648,-3.604102,-0.647316,0.001119
1,AAL3_026_OFCmed_R,AAL3_078_Putamen_R,-0.589434,1.865329,0.156725,1.773788,0.746159,0.195418,1.296901,2.766924,0.496954,0.009597


...

,ROI_1,ROI_2,pre_deviation,pre_SD_deviation,post_deviation,post_SD_deviation,post_minus_pre,post_minus_pre_CI95_low,post_minus_pre_CI95_high,paired_t,effect_size_Cohen_dz,paired_p_uncorrected
0,AAL3_022_Frontal_Med_Orb_R,AAL3_078_Putamen_R,-0.609059,1.408988,0.200815,1.458174,0.809874,0.425701,1.194047,4.325453,0.817434,0.000186
1,AAL3_022_Frontal_Med_Orb_R,AAL3_076_Caudate_R,-1.939631,2.403259,-0.782990,1.559634,1.156641,0.428986,1.884297,3.261476,0.616361,0.002999
2,AAL3_026_OFCmed_R,AAL3_078_Putamen_R,-0.538597,1.514265,0.173469,1.557531,0.712066,0.254099,1.170032,3.190274,0.602905,0.003586
3,AAL3_005_Frontal_Mid_2_L,AAL3_080_Pallidum_R,0.445601,2.067803,-0.616486,1.620629,-1.062087,-1.805427,-0.318748,-2.931667,-0.554033,0.006789
4,AAL3_021_Frontal_Med_Orb_L,AAL3_076_Caudate_R,-2.540373,3.242317,-1.088782,2.555716,1.451591,0.430548,2.472635,2.917035,0.551268,0.007034
5,AAL3_021_Frontal_Med_Orb_L,AAL3_075_Caudate_L,-0.664727,1.592166,0.058507,1.530491,0.723234,0.207433,1.239034,2.876991,0.543700,0.007750


**iTBS HDRS-17 associations**  
`outputs/main_analyses/02_TRD2_deviation_prepost_agesex/03_iTBS_prepost/edge_HDRS_partial_spearman_all_prepost_p_lt_0p05.xlsx` — 15 rows × 18 columns

,ROI_1,ROI_2,pre_deviation,pre_SD_deviation,post_deviation,post_SD_deviation,post_minus_pre,post_minus_pre_CI95_low,post_minus_pre_CI95_high,paired_t,effect_size_Cohen_dz,paired_p_uncorrected,HDRS_partial_Spearman_r,HDRS_p_uncorrected,n_used,HDRS_q_fdr_BH_within_prepost_significant_edges,HDRS_primary_FDR_q_lt_0p05,HDRS_exploratory_uncorrected_p_lt_0p05
0,AAL3_003_Frontal_Sup_2_L,AAL3_043_ParaHippocampal_L,-0.333037,0.956534,-0.781479,1.576595,-0.448442,-0.824295,-0.072590,-2.436703,-0.437645,0.020969,0.349734,0.062920,31,0.395767,False,False
1,AAL3_026_OFCmed_R,AAL3_078_Putamen_R,-0.589434,1.865329,0.156725,1.773788,0.746159,0.195418,1.296901,2.766924,0.496954,0.009597,-0.339926,0.071201,31,0.395767,False,False
2,AAL3_020_Frontal_Sup_Medial_R,AAL3_151_ACC_sub_L,-0.739533,1.702727,-1.365939,2.033472,-0.626407,-1.164504,-0.088310,-2.377440,-0.427001,0.024008,0.331305,0.079153,31,0.395767,False,False
3,AAL3_003_Frontal_Sup_2_L,AAL3_046_Amygdala_R,0.246742,1.588659,-0.373249,1.660834,-0.619991,-1.215889,-0.024093,-2.124845,-0.381633,0.041951,0.256661,0.178946,31,0.559270,False,False
4,AAL3_005_Frontal_Mid_2_L,AAL3_042_Hippocampus_R,0.292911,2.060205,-0.448726,2.093085,-0.741637,-1.428218,-0.055056,-2.206038,-0.396216,0.035181,0.251095,0.188888,31,0.559270,False,False
5,AAL3_026_OFCmed_R,AAL3_080_Pallidum_R,-1.223280,2.041005,-0.559661,1.922030,0.663619,0.122167,1.205071,2.503068,0.449564,0.017987,-0.230294,0.229418,31,0.559270,False,False
6,AAL3_005_Frontal_Mid_2_L,AAL3_043_ParaHippocampal_L,-0.214279,1.104987,-0.693165,1.894068,-0.478886,-0.873582,-0.084190,-2.477894,-0.445043,0.019069,0.215751,0.260993,31,0.559270,False,False
7,AAL3_031_OFClat_L,AAL3_041_Hippocampus_L,-0.194847,0.644053,-0.482488,0.819397,-0.287641,-0.450633,-0.124648,-3.604102,-0.647316,0.001119,0.122463,0.526821,31,0.821264,False,False
8,AAL3_006_Frontal_Mid_2_R,AAL3_079_Pallidum_L,-2.314258,3.003016,-1.599745,3.093477,0.714513,0.091649,1.337376,2.342777,0.420775,0.025966,-0.114684,0.553593,31,0.821264,False,False
9,AAL3_020_Frontal_Sup_Medial_R,AAL3_043_ParaHippocampal_L,-0.379903,0.961830,-0.830415,1.593932,-0.450513,-0.805999,-0.095027,-2.588204,-0.464855,0.014733,0.111004,0.566472,31,0.821264,False,False


**rTMS HDRS-17 associations**  
`outputs/main_analyses/02_TRD2_deviation_prepost_agesex/04_rTMS_prepost/edge_HDRS_partial_spearman_all_prepost_p_lt_0p05.xlsx` — 13 rows × 18 columns

,ROI_1,ROI_2,pre_deviation,pre_SD_deviation,post_deviation,post_SD_deviation,post_minus_pre,post_minus_pre_CI95_low,post_minus_pre_CI95_high,paired_t,effect_size_Cohen_dz,paired_p_uncorrected,HDRS_partial_Spearman_r,HDRS_p_uncorrected,n_used,HDRS_q_fdr_BH_within_prepost_significant_edges,HDRS_primary_FDR_q_lt_0p05,HDRS_exploratory_uncorrected_p_lt_0p05
0,AAL3_022_Frontal_Med_Orb_R,AAL3_080_Pallidum_R,0.314397,1.390389,0.733479,1.649664,0.419082,0.024189,0.813975,2.177514,0.411512,0.038353,0.513427,0.007305,28,0.094969,False,True
1,AAL3_005_Frontal_Mid_2_L,AAL3_080_Pallidum_R,0.445601,2.067803,-0.616486,1.620629,-1.062087,-1.805427,-0.318748,-2.931667,-0.554033,0.006789,0.396327,0.045024,28,0.292654,False,True
2,AAL3_021_Frontal_Med_Orb_L,AAL3_075_Caudate_L,-0.664727,1.592166,0.058507,1.530491,0.723234,0.207433,1.239034,2.876991,0.543700,0.007750,0.357868,0.072651,28,0.314821,False,False
3,AAL3_026_OFCmed_R,AAL3_077_Putamen_L,-0.668595,1.269663,-0.248712,1.162291,0.419882,0.041983,0.797781,2.279783,0.430838,0.030740,0.246945,0.223911,28,0.672250,False,False
4,AAL3_025_OFCmed_L,AAL3_079_Pallidum_L,-0.212753,0.892385,-0.666389,1.606766,-0.453636,-0.869521,-0.037750,-2.238075,-0.422956,0.033662,-0.198559,0.330857,28,0.672250,False,False
5,AAL3_005_Frontal_Mid_2_L,AAL3_078_Putamen_R,-0.910053,2.592642,-1.542879,2.627277,-0.632826,-1.253711,-0.011940,-2.091290,-0.395217,0.046044,0.193148,0.344468,28,0.672250,False,False
6,AAL3_026_OFCmed_R,AAL3_042_Hippocampus_R,-0.113943,0.708939,0.214139,0.996073,0.328082,0.011831,0.644334,2.128587,0.402265,0.042563,0.186371,0.361981,28,0.672250,False,False
7,AAL3_026_OFCmed_R,AAL3_076_Caudate_R,-0.469101,1.246135,0.078167,1.170601,0.547267,0.092717,1.001817,2.470353,0.466853,0.020105,0.161918,0.429380,28,0.674214,False,False
8,AAL3_022_Frontal_Med_Orb_R,AAL3_076_Caudate_R,-1.939631,2.403259,-0.782990,1.559634,1.156641,0.428986,1.884297,3.261476,0.616361,0.002999,0.138656,0.499349,28,0.674214,False,False
9,AAL3_026_OFCmed_R,AAL3_078_Putamen_R,-0.538597,1.514265,0.173469,1.557531,0.712066,0.254099,1.170032,3.190274,0.602905,0.003586,0.128446,0.531750,28,0.674214,False,False
